# 02 - Feature EngineeringTurning a raw time series into a supervised-learning problem, **without leakingthe future into the past**.The rule enforced throughout: *every feature must be computable at the momentthe prediction is made.*

In [ ]:
import sysfrom pathlib import Path# Make the project importable when the notebook runs from notebooks/ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT))import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snssns.set_theme(style="whitegrid")plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
from src.data.preprocess import load_processed_datafrom src.features.feature_engineering import (    add_time_features, add_lag_features, add_rolling_features,    create_features, get_feature_columns, build_training_frame,)from src.utils import configdf = load_processed_data()print(f"{len(df):,} samples from {df['timestamp'].min()} to {df['timestamp'].max()}")

## 1. Calendar featuresKnown in advance for any future timestamp, so they can never leak.Hour and month are *also* encoded as sine/cosine pairs. Reasons:1. Hour 23 and hour 0 are adjacent in time but 23 units apart as integers.2. Tree models cannot extrapolate beyond values seen in training. The test month   (September = 9) never appears in the training months (1-8), but its sin/cos   values fall inside the range already observed, so the model interpolates.

In [ ]:
timed = add_time_features(df)timed[["timestamp", "hour", "day_of_week", "is_weekend",       "hour_sin", "hour_cos", "month_sin", "month_cos"]].head()

In [ ]:
one_day = timed.head(24)plt.plot(one_day["hour"], one_day["hour_sin"], "o-", label="hour_sin")plt.plot(one_day["hour"], one_day["hour_cos"], "s-", label="hour_cos")plt.title("Cyclical encoding of the hour"); plt.xlabel("hour"); plt.legend(); plt.show()

## 2. Lag features`lag_k` is the target k intervals earlier. With hourly sampling `lag_24` is thesame hour yesterday — usually the single strongest predictor for a daily cycle.

In [ ]:
lagged = add_lag_features(df)lagged[["timestamp", "active_power", "lag_1", "lag_2", "lag_3", "lag_24"]].head(30).tail(8)

In [ ]:
# Verify the shift direction: lag_1 at row i must equal the target at row i-1i = 100assert lagged["lag_1"].iloc[i] == lagged["active_power"].iloc[i - 1]assert lagged["lag_24"].iloc[i] == lagged["active_power"].iloc[i - 24]print("Lag alignment verified - lags look strictly backwards in time")

## 3. Rolling statistics — the leakage trap`df[target].rolling(3).mean()` includes the **current** sample, so the featurewould contain the answer. The correct form shifts first:```pythondf[target].shift(1).rolling(window).mean()```

In [ ]:
rolled = add_rolling_features(df)window = 3i = 100correct = rolled["rolling_mean_3"].iloc[i]manual_past_only = df["active_power"].iloc[i - window:i].mean()leaky = df["active_power"].iloc[i - window + 1:i + 1].mean()print(f"implementation        : {correct:.3f}")print(f"past-only (expected)  : {manual_past_only:.3f}")print(f"leaky (includes now)  : {leaky:.3f}")

## 4. Which columns become model inputs?Excluded deliberately:| Column | Why it is excluded ||---|---|| `active_power` | it is the target || `energy` | E = P·Δt/1000, a rescaled copy of the target || `voltage`, `current`, `power_factor`, `frequency` | measured at the *same instant* as the target and tied to it by P = V·I·PF; unavailable when forecasting a future hour || `timestamp` | not numeric; already encoded as calendar features |

In [ ]:
featured = create_features(df)features = get_feature_columns(featured)print(f"{len(features)} features:")for f in features:    print("  -", f)print(f"\nWarm-up rows dropped: {len(df) - len(featured)} "      f"(max lag/window = {max(config.LAG_PERIODS + config.ROLLING_WINDOWS)})")

## 5. What a leaky feature set looks likeIf `current` is included, the model reaches an R² near 1.0 — not because itforecasts well, but because it is reading the answer off the input. This is themost important cell in the notebook.

In [ ]:
from sklearn.ensemble import RandomForestRegressorfrom sklearn.metrics import r2_scoreX, y, names, ts = build_training_frame(df)cut = int(len(X) * 0.8)honest = RandomForestRegressor(n_estimators=60, random_state=42, n_jobs=-1)honest.fit(X.iloc[:cut], y.iloc[:cut])r2_honest = r2_score(y.iloc[cut:], honest.predict(X.iloc[cut:]))X_leaky = X.copy()X_leaky["current"] = featured["current"].to_numpy()      # deliberate mistakeleaky = RandomForestRegressor(n_estimators=60, random_state=42, n_jobs=-1)leaky.fit(X_leaky.iloc[:cut], y.iloc[:cut])r2_leaky = r2_score(y.iloc[cut:], leaky.predict(X_leaky.iloc[cut:]))print(f"Honest feature set : R2 = {r2_honest:.4f}")print(f"Leaky  feature set : R2 = {r2_leaky:.4f}   <-- too good to be true")

## 6. Feature importance (sanity check)

In [ ]:
importance = pd.Series(honest.feature_importances_, index=names).sort_values()importance.plot(kind="barh", figsize=(8, 5), title="Random Forest feature importance")plt.tight_layout(); plt.show()

## Takeaways- Lags and shifted rolling statistics carry most of the signal.- Concurrent electrical measurements are excluded on purpose.- A suspiciously high R² is a leakage symptom, not a success.